# PyProgressive — Progressive Visualization (Phase 1)

This notebook demonstrates `pp.viz`, which binds PyProgressive's progressive computation to interactive Plotly charts.

Two chart types are available:
- **`pp.viz.line()`** — each variable becomes a line; x-axis = elapsed time, y-axis = current value
- **`pp.viz.scatter()`** — shows the convergence *trajectory* of two variables as a moving point

In [ ]:
%pip install -e ..
%pip install plotly anywidget

---
## 1. Line Chart — Basic Example

Compute mean and variance of a small array progressively.
The chart updates in real-time as each data point is processed.

In [ ]:
import pyprogressive as pp
from pyprogressive import each, accum

pp.reset()  # clear arrays from previous cells

data = pp.array([10, 20, 0, 21, 5, 42, 11, 14, 34, 13])

mean = accum(each(data)) / len(data)
var  = accum((each(data) - mean) ** 2) / len(data)

program = pp.compile(mean, var)

chart = pp.viz.line(
    labels=["Mean", "Variance"],
    title="Progressive Mean & Variance"
)
chart.run(program, interval=0.0001)

---
## 2. Line Chart — California Housing Dataset

Compute covariance and variance for multiple features progressively.
Each statistic is one line; watch them converge as more data is processed.

In [ ]:
import pyprogressive as pp
from pyprogressive import each, accum
from sklearn.datasets import fetch_california_housing

pp.reset()  # clear arrays from previous cells

data = fetch_california_housing(as_frame=True)
X = data.data
y = data.target

arrayX1 = pp.array(X["MedInc"].tolist())
arrayX2 = pp.array(X["HouseAge"].tolist())
arrayY  = pp.array(y.tolist())

mean1  = accum(each(arrayX1)) / len(arrayX1)
mean2  = accum(each(arrayX2)) / len(arrayX2)
meanY  = accum(each(arrayY))  / len(arrayY)

var1   = accum((each(arrayX1) - mean1) ** 2) / len(arrayX1)
var2   = accum((each(arrayX2) - mean2) ** 2) / len(arrayX2)

covX1Y = accum((each(arrayX1) - mean1) * (each(arrayY) - meanY)) / len(arrayX1)
covX2Y = accum((each(arrayX2) - mean2) * (each(arrayY) - meanY)) / len(arrayX2)

program = pp.compile(covX1Y, covX2Y, var1, var2)

chart = pp.viz.line(
    labels=["Cov(MedInc, Price)", "Cov(HouseAge, Price)", "Var(MedInc)", "Var(HouseAge)"],
    title="California Housing — Progressive Statistics"
)
chart.run(program, interval=0.3)

---
## 3. Scatter Chart — Convergence Trajectory

`pp.viz.scatter()` takes exactly 2 variables.

The **blue line** is the trajectory of `(var0, var1)` over time;  
the **red dot** is the current estimate.  
As more data is processed, the dot stabilises at the true value.

In [ ]:
import pyprogressive as pp
from pyprogressive import each, accum
from sklearn.datasets import fetch_california_housing

pp.reset()  # clear arrays from previous cells

data = fetch_california_housing(as_frame=True)
X = data.data
y = data.target

arrayX1 = pp.array(X["MedInc"].tolist())
arrayY  = pp.array(y.tolist())

mean1 = accum(each(arrayX1)) / len(arrayX1)
meanY = accum(each(arrayY))  / len(arrayY)

var1   = accum((each(arrayX1) - mean1) ** 2) / len(arrayX1)
covX1Y = accum((each(arrayX1) - mean1) * (each(arrayY) - meanY)) / len(arrayX1)

program = pp.compile(covX1Y, var1)

chart = pp.viz.scatter(
    x_label="Cov(MedInc, Price)",
    y_label="Var(MedInc)",
    title="Convergence Trajectory — Cov vs Var"
)
chart.run(program, interval=0.3)